In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose          import ColumnTransformer
from sklearn.preprocessing    import StandardScaler, OneHotEncoder
from sklearn.decomposition    import PCA
from sklearn.linear_model     import LogisticRegression
from sklearn.tree             import DecisionTreeClassifier
from sklearn.pipeline         import Pipeline
from sklearn.model_selection  import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics          import accuracy_score, classification_report, roc_auc_score

# ── 1. Sensor ────────────────────────────────────────────────────────────
class TransactionSensor:
    def __init__(self, path: str):
        self.path = path

    def stream(self) -> pd.DataFrame:
        print(f"[Sensor] Reading file {self.path} …")
        df = pd.read_csv(self.path)
        print(f"[Sensor] Loaded {len(df)} rows × {df.shape[1]} cols\n")
        return df

# ── 2. Perception ────────────────────────────────────────────────────────
class PerceptionModule:
    def __init__(self, num_raw, cat_cols):
        self.num_raw  = num_raw
        self.cat_cols = cat_cols
        self.scaler   = StandardScaler()
        print("[Perception] Initializing StandardScaler and OneHotEncoder+PCA pipeline")

        self.prep = ColumnTransformer([
            ('num', 'passthrough', [c + '_z' for c in num_raw]),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
        ])
        self.pca  = PCA(n_components=0.95, random_state=42)
        self.pipe = Pipeline([('prep', self.prep), ('pca', self.pca)])

    def fit(self, X: pd.DataFrame):
        print("[Perception] Fitting scaler on numeric columns:")
        X_num = X[self.num_raw]
        X_z   = self.scaler.fit_transform(X_num)
        for i, c in enumerate(self.num_raw):
            print(f"  • Scaling {c}: mean={self.scaler.mean_[i]:.2f}, std={self.scaler.scale_[i]:.2f}")
            X[c + '_z'] = X_z[:, i] # This line modifies the original X (X_train)
        print("\n[Perception] Fitting PCA on preprocessed data…")
        self.pipe.fit(X)
        cumvar = np.cumsum(self.pca.explained_variance_ratio_)
        print(f"  • PCA kept {self.pca.n_components_} components")
        print(f"  • Cumulative variance explained: {cumvar[-1]:.3f}\n")

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        print("[Perception] Transforming data through PCA pipeline…")
        # Create the '_z' suffixed columns in X (X_test) before transforming
        X_num = X[self.num_raw]
        X_z   = self.scaler.transform(X_num) # Use transform, not fit_transform
        for i, c in enumerate(self.num_raw):
            X[c + '_z'] = X_z[:, i]
        X_pca = self.pipe.transform(X)
        print(f"  • Output shape: {X_pca.shape}")
        print(f"  • Sample PC1 values: {np.round(X_pca[:5,0],3)}\n")
        return X_pca

# ── 3a. CognitiveCore – Logistic Regression ───────────────────────────────
class LogisticCore:
    def __init__(self):
        print("[Cognition:LogReg] Initializing Logistic Regression model")
        self.clf = LogisticRegression(max_iter=1000, class_weight='balanced', solver='lbfgs')

    def fit(self, X, y):
        print("[Cognition:LogReg] Training Logistic Regression…")
        self.clf.fit(X, y)

    def predict(self, X):
        print("[Cognition:LogReg] Predicting with Logistic Regression…")
        return self.clf.predict(X)

    def score(self, X, y):
        acc = accuracy_score(y, self.predict(X))
        print(f"[Cognition:LogReg] Accuracy = {acc:.3f}\n")
        return acc

# ── 3b. CognitiveCore – Decision Tree ────────────────────────────────────
class TreeCore:
    def __init__(self):
        print("[Cognition:Tree] Initializing Decision Tree model")
        self.clf = DecisionTreeClassifier(random_state=42, class_weight='balanced')

    def fit(self, X, y):
        print("[Cognition:Tree] Training Decision Tree…")
        self.clf.fit(X, y)

    def predict(self, X):
        print("[Cognition:Tree] Predicting with Decision Tree…")
        return self.clf.predict(X)

    def score(self, X, y):
        acc = accuracy_score(y, self.predict(X))
        print(f"[Cognition:Tree] Accuracy = {acc:.3f}\n")
        return acc

# ── 4. Actuator ──────────────────────────────────────────────────────────
class FraudActuator:
    @staticmethod
    def alert(model_name, ids, verdicts):
        print(f"[Actuator:{model_name}] Alerting flagged transactions:")
        for tx_id, flag in zip(ids, verdicts):
            if flag:
                print(f"  >> Transaction {tx_id} flagged as FRAUD")
        print()

# ── 5. Main Flow ─────────────────────────────────────────────────────────
# ── 5. Main Flow ─────────────────────────────────────────────────────────
if __name__ == "__main__":
    # 5.1 Sensor
    sensor = TransactionSensor("data_labeled.csv")
    df     = sensor.stream()

    # 5.2 Split
    print("[Main] Splitting features and label…\n")
    y     = df['Class']
    Xraw  = df.drop(columns='Class')

    # 5.3 Define columns
    num_raw  = ['TransactionAmount','AccountBalance','TransactionDuration','CustomerAge','LoginAttempts']
    cat_cols = ['Location','Channel']

    # 5.4 Train/Test split
    print("[Main] Performing train-test split (80/20)…\n")
    X_train, X_test, y_train, y_test = train_test_split(
        Xraw, y, test_size=0.2, stratify=y, random_state=42)

    # 5.5 Perception
    vision = PerceptionModule(num_raw, cat_cols)
    vision.fit(X_train)
    X_train_pca = vision.transform(X_train)
    X_test_pca  = vision.transform(X_test)

    # 5.6 Baseline heuristic
    print("[Baseline] Computing PC1 threshold heuristic…")
    threshold   = np.median(X_train_pca[:,0])
    y_pred_heur = (X_test_pca[:,0] > threshold).astype(int)
    print(f"  • PC1 threshold = {threshold:.3f}")
    print(f"  • Heuristic accuracy = {accuracy_score(y_test, y_pred_heur):.3f}\n")

    # 5.7 Cognition + Actuation – Logistic Regression
    log_core = LogisticCore()
    log_core.fit(X_train_pca, y_train)
    y_pred_lr = log_core.predict(X_test_pca)
    log_core.score(X_test_pca, y_test)
    FraudActuator.alert("LogReg", X_test['TransactionID'].values, y_pred_lr)

    # 5.8 Cognition + Actuation – Decision Tree
    tree_core = TreeCore()
    tree_core.fit(X_train_pca, y_train)
    y_pred_dt = tree_core.predict(X_test_pca)
    tree_core.score(X_test_pca, y_test)
    FraudActuator.alert("DecTree", X_test['TransactionID'].values, y_pred_dt)

    # 5.9 Cross-validation comparison
# 5.9 Cross-validation comparison
print("[Main] Running 5-fold CV for accuracy comparison…")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# a) Logistic Regression
# Create a new pipeline for cross-validation to avoid using the fitted vision.pipe
pipe_lr = Pipeline([
    ('prep', ColumnTransformer([
        ('num', StandardScaler(), num_raw),  # Apply StandardScaler directly
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ])),
    ('pca', PCA(n_components=0.95, random_state=42)),  # Include PCA
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', solver='lbfgs'))
])
scores_lr = cross_val_score(pipe_lr, Xraw, y,
                              scoring='accuracy',
                              cv=cv,
                              n_jobs=-1)   # parallel
print(f"  • LogReg CV accuracies per fold: {np.round(scores_lr, 3)}")
print(f"  • Mean LogReg CV accuracy: {scores_lr.mean():.3f}")

# b) Decision Tree
# Create a new pipeline for Decision Tree cross-validation
pipe_dt = Pipeline([
    ('prep', ColumnTransformer([
        ('num', StandardScaler(), num_raw),  # Apply StandardScaler directly
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ])),
    ('pca', PCA(n_components=0.95, random_state=42)),  # Include PCA
    ('clf', DecisionTreeClassifier(random_state=42, class_weight='balanced'))
])
scores_dt = cross_val_score(pipe_dt, Xraw, y,
                              scoring='accuracy',
                              cv=cv,
                              n_jobs=-1)
print(f"  • DecTree CV accuracies per fold: {np.round(scores_dt, 3)}")
print(f"  • Mean DecTree CV accuracy: {scores_dt.mean():.3f}\n")

print("[Main] Completed all tasks!")

[Sensor] Reading file data_labeled.csv …
[Sensor] Loaded 2512 rows × 17 cols

[Main] Splitting features and label…

[Main] Performing train-test split (80/20)…

[Perception] Initializing StandardScaler and OneHotEncoder+PCA pipeline
[Perception] Fitting scaler on numeric columns:
  • Scaling TransactionAmount: mean=297.70, std=289.97
  • Scaling AccountBalance: mean=5111.21, std=3930.22
  • Scaling TransactionDuration: mean=119.92, std=69.80
  • Scaling CustomerAge: mean=44.85, std=17.90
  • Scaling LoginAttempts: mean=1.13, std=0.61

[Perception] Fitting PCA on preprocessed data…
  • PCA kept 49 components
  • Cumulative variance explained: 0.950

[Perception] Transforming data through PCA pipeline…
  • Output shape: (2009, 49)
  • Sample PC1 values: [-1.415  1.56   0.777  0.732 -0.264]

[Perception] Transforming data through PCA pipeline…
  • Output shape: (503, 49)
  • Sample PC1 values: [ 1.169  0.514 -0.326 -0.305  0.503]

[Baseline] Computing PC1 threshold heuristic…
  • PC1 thre

In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose          import ColumnTransformer
from sklearn.preprocessing    import StandardScaler, OneHotEncoder
from sklearn.decomposition    import PCA
from sklearn.linear_model     import LogisticRegression
from sklearn.tree             import DecisionTreeClassifier
from sklearn.pipeline         import Pipeline
from sklearn.model_selection  import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics          import accuracy_score, classification_report, roc_auc_score
from sklearn.metrics import precision_score, recall_score, f1_score

# ── 1. Sensor ────────────────────────────────────────────────────────────
class TransactionSensor:
    def __init__(self, path: str):
        self.path = path

    def stream(self) -> pd.DataFrame:
        print(f"[Sensor] Reading file {self.path} …")
        df = pd.read_csv(self.path)
        print(f"[Sensor] Loaded {len(df)} rows × {df.shape[1]} cols\n")
        return df

# ── 2. Perception ────────────────────────────────────────────────────────
class PerceptionModule:
    def __init__(self, num_raw, cat_cols):
        self.num_raw  = num_raw
        self.cat_cols = cat_cols
        self.scaler   = StandardScaler()
        print("[Perception] Initializing StandardScaler and OneHotEncoder+PCA pipeline")

        self.prep = ColumnTransformer([
            ('num', 'passthrough', [c + '_z' for c in num_raw]),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
        ])
        self.pca  = PCA(n_components=0.95, random_state=42)
        self.pipe = Pipeline([('prep', self.prep), ('pca', self.pca)])

    def fit(self, X: pd.DataFrame):
        print("[Perception] Fitting scaler on numeric columns:")
        X_num = X[self.num_raw]
        X_z   = self.scaler.fit_transform(X_num)
        for i, c in enumerate(self.num_raw):
            print(f"  • Scaling {c}: mean={self.scaler.mean_[i]:.2f}, std={self.scaler.scale_[i]:.2f}")
            X[c + '_z'] = X_z[:, i] # This line modifies the original X (X_train)
        print("\n[Perception] Fitting PCA on preprocessed data…")
        self.pipe.fit(X)
        cumvar = np.cumsum(self.pca.explained_variance_ratio_)
        print(f"  • PCA kept {self.pca.n_components_} components")
        print(f"  • Cumulative variance explained: {cumvar[-1]:.3f}\n")

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        print("[Perception] Transforming data through PCA pipeline…")
        # Create the '_z' suffixed columns in X (X_test) before transforming
        X_num = X[self.num_raw]
        X_z   = self.scaler.transform(X_num) # Use transform, not fit_transform
        for i, c in enumerate(self.num_raw):
            X[c + '_z'] = X_z[:, i]
        X_pca = self.pipe.transform(X)
        print(f"  • Output shape: {X_pca.shape}")
        print(f"  • Sample PC1 values: {np.round(X_pca[:5,0],3)}\n")
        return X_pca

# ── 3a. CognitiveCore – Logistic Regression ───────────────────────────────
class LogisticCore:
    def __init__(self):
        print("[Cognition:LogReg] Initializing Logistic Regression model")
        self.clf = LogisticRegression(max_iter=1000, class_weight='balanced', solver='lbfgs')

    def fit(self, X, y):
        print("[Cognition:LogReg] Training Logistic Regression…")
        self.clf.fit(X, y)

    def predict(self, X):
        print("[Cognition:LogReg] Predicting with Logistic Regression…")
        return self.clf.predict(X)

    def score(self, X, y):
        acc = accuracy_score(y, self.predict(X))
        print(f"[Cognition:LogReg] Accuracy = {acc:.3f}\n")
        return acc

# ── 3b. CognitiveCore – Decision Tree ────────────────────────────────────
class TreeCore:
    def __init__(self):
        print("[Cognition:Tree] Initializing Decision Tree model")
        self.clf = DecisionTreeClassifier(random_state=42, class_weight='balanced')

    def fit(self, X, y):
        print("[Cognition:Tree] Training Decision Tree…")
        self.clf.fit(X, y)

    def predict(self, X):
        print("[Cognition:Tree] Predicting with Decision Tree…")
        return self.clf.predict(X)

    def score(self, X, y):
        acc = accuracy_score(y, self.predict(X))
        print(f"[Cognition:Tree] Accuracy = {acc:.3f}\n")
        return acc

# ── 4. Actuator ──────────────────────────────────────────────────────────
class FraudActuator:
    @staticmethod
    def alert(model_name, ids, verdicts):
        print(f"[Actuator:{model_name}] Alerting flagged transactions:")
        for tx_id, flag in zip(ids, verdicts):
            if flag:
                print(f"  >> Transaction {tx_id} flagged as FRAUD")
        print()

# ── 5. Main Flow ─────────────────────────────────────────────────────────
if __name__ == "__main__":
    # 5.1 Sensor
    sensor = TransactionSensor("data_labeled.csv")
    df     = sensor.stream()

    # 5.2 Split
    print("[Main] Splitting features and label…\n")
    y     = df['Class']
    Xraw  = df.drop(columns='Class')

    # 5.3 Define columns
    num_raw  = ['TransactionAmount','AccountBalance','TransactionDuration','CustomerAge','LoginAttempts']
    cat_cols = ['Location','Channel']

    # 5.4 Train/Test split
    print("[Main] Performing train-test split (80/20)…\n")
    X_train, X_test, y_train, y_test = train_test_split(
        Xraw, y, test_size=0.2, stratify=y, random_state=42)

    # 5.5 Perception
    vision = PerceptionModule(num_raw, cat_cols)
    vision.fit(X_train)
    X_train_pca = vision.transform(X_train)
    X_test_pca  = vision.transform(X_test)

    # 5.6 Baseline heuristic
    print("[Baseline] Computing PC1 threshold heuristic…")
    threshold   = np.median(X_train_pca[:,0])
    y_pred_heur = (X_test_pca[:,0] > threshold).astype(int)
    print(f"  • PC1 threshold = {threshold:.3f}")
    print(f"  • Heuristic accuracy = {accuracy_score(y_test, y_pred_heur):.3f}\n")

    # 5.7 Cognition + Actuation – Logistic Regression
    log_core = LogisticCore()
    log_core.fit(X_train_pca, y_train)
    y_pred_lr = log_core.predict(X_test_pca)
    log_core.score(X_test_pca, y_test)
    FraudActuator.alert("LogReg", X_test['TransactionID'].values, y_pred_lr)

    # Calculate precision, recall, and F1-score for Logistic Regression
    precision_lr = precision_score(y_test, y_pred_lr)
    recall_lr = recall_score(y_test, y_pred_lr)
    f1_lr = f1_score(y_test, y_pred_lr)
    print(f"[Cognition:LogReg] Precision = {precision_lr:.3f}")
    print(f"[Cognition:LogReg] Recall = {recall_lr:.3f}")
    print(f"[Cognition:LogReg] F1-score = {f1_lr:.3f}\n")

    # 5.8 Cognition + Actuation – Decision Tree
    tree_core = TreeCore()
    tree_core.fit(X_train_pca, y_train)
    y_pred_dt = tree_core.predict(X_test_pca)
    tree_core.score(X_test_pca, y_test)
    FraudActuator.alert("DecTree", X_test['TransactionID'].values, y_pred_dt)

    # Calculate precision, recall, and F1-score for Decision Tree
    precision_dt = precision_score(y_test, y_pred_dt)
    recall_dt = recall_score(y_test, y_pred_dt)
    f1_dt = f1_score(y_test, y_pred_dt)
    print(f"[Cognition:Tree] Precision = {precision_dt:.3f}")
    print(f"[Cognition:Tree] Recall = {recall_dt:.3f}")
    print(f"[Cognition:Tree] F1-score = {f1_dt:.3f}\n")

    # 5.9 Cross-validation comparison
    print("[Main] Running 5-fold CV for accuracy comparison…")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # a) Logistic Regression
    # Create a new pipeline for cross-validation to avoid using the fitted vision.pipe
    pipe_lr = Pipeline([
        ('prep', ColumnTransformer([
            ('num', StandardScaler(), num_raw),  # Apply StandardScaler directly
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
        ])),
        ('pca', PCA(n_components=0.95, random_state=42)),  # Include PCA
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', solver='lbfgs'))
    ])
    scores_lr = cross_val_score(pipe_lr, Xraw, y,
                                  scoring='accuracy',
                                  cv=cv,
                                  n_jobs=-1)   # parallel
    print(f"  • LogReg CV accuracies per fold: {np.round(scores_lr, 3)}")
    print(f"  • Mean LogReg CV accuracy: {scores_lr.mean():.3f}")

    # b) Decision Tree
    # Create a new pipeline for Decision Tree cross-validation
    pipe_dt = Pipeline([
        ('prep', ColumnTransformer([
            ('num', StandardScaler(), num_raw),  # Apply StandardScaler directly
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
        ])),
        ('pca', PCA(n_components=0.95, random_state=42)),  # Include PCA
        ('clf', DecisionTreeClassifier(random_state=42, class_weight='balanced'))
    ])
    scores_dt = cross_val_score(pipe_dt, Xraw, y,
                                  scoring='accuracy',
                                  cv=cv,
                                  n_jobs=-1)
    print(f"  • DecTree CV accuracies per fold: {np.round(scores_dt, 3)}")
    print(f"  • Mean DecTree CV accuracy: {scores_dt.mean():.3f}\n")

    print("[Main] Completed all tasks!")

[Sensor] Reading file data_labeled.csv …
[Sensor] Loaded 2512 rows × 17 cols

[Main] Splitting features and label…

[Main] Performing train-test split (80/20)…

[Perception] Initializing StandardScaler and OneHotEncoder+PCA pipeline
[Perception] Fitting scaler on numeric columns:
  • Scaling TransactionAmount: mean=297.70, std=289.97
  • Scaling AccountBalance: mean=5111.21, std=3930.22
  • Scaling TransactionDuration: mean=119.92, std=69.80
  • Scaling CustomerAge: mean=44.85, std=17.90
  • Scaling LoginAttempts: mean=1.13, std=0.61

[Perception] Fitting PCA on preprocessed data…
  • PCA kept 49 components
  • Cumulative variance explained: 0.950

[Perception] Transforming data through PCA pipeline…
  • Output shape: (2009, 49)
  • Sample PC1 values: [-1.415  1.56   0.777  0.732 -0.264]

[Perception] Transforming data through PCA pipeline…
  • Output shape: (503, 49)
  • Sample PC1 values: [ 1.169  0.514 -0.326 -0.305  0.503]

[Baseline] Computing PC1 threshold heuristic…
  • PC1 thre